In [ ]:
#!pip install git+https://github.com/openai/whisper.git
#!sudo apt update && sudo apt install ffmpeg
#!whisper "/voice-sample.wav" --model base.en

#####---TESTING---#####

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-jrbk5t5t
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-jrbk5t5t
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ub

In [ ]:
!pip install git+https://github.com/openai/whisper.git
!sudo apt update && sudo apt install ffmpeg

import whisper
from whisper.utils import get_writer
import os

# Load the model (you can choose "tiny", "base", "small", "medium", or "large")
model = whisper.load_model("base")

# Define the path to your audio file (upload it to Colab first and copy its path)
# Example path: "/content/audio.mp3"
audio_file_path = "/content/voice-sample.wav"

# Transcribe the audio
result = model.transcribe(audio_file_path)

# Specify the output directory
output_directory = "/content/"

# --- Generate file with timestamps (.srt format) ---
srt_writer = get_writer("srt", output_directory)
srt_writer(result, audio_file_path.split('.')[0], {"max_line_width": None, "max_line_count": None, "highlight_words": False})

print(f"SRT file saved to {output_directory}{audio_file_path.split('.')[0]}.srt")

# --- Generate plain text file (.txt format) ---
txt_writer = get_writer("txt", output_directory)
txt_writer(result, audio_file_path.split('.')[0], {})

print(f"TXT file saved to {output_directory}{audio_file_path.split('.')[0]}.txt")



In [ ]:

!apt-get update && apt-get install -y zstd
!pip install langchain_community langchain_ollama

# Install Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!ollama --version

In [ ]:
import subprocess
import threading
import time

# Start the Ollama server manually in the background
def run_ollama_serve():
    subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

# Wait for the server to wake up
time.sleep(10)

# Pull the model (this might take 2-3 minutes)
!ollama pull llama3
!ollama pull llama3.2



In [ ]:
!ollama list    #CHECK OLAMA RUNNING


In [ ]:
# 1. Install (Run this once)
!pip install langchain-text-splitters langchain-ollama langchain-core

# 2. Correct Imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate # <--- This is the correct one

# 3. Initialize the Llama3 model
# Make sure you ran !ollama pull llama3.2 earlier!
llm = OllamaLLM(model="llama3.2")

def split_content(text):
    """Splits long text into manageable chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100
    )
    return text_splitter.split_text(text)

def summarize_section(chunks):
    """Summarizes each chunk using Llama3."""
    summaries = []
    for chunk in chunks:
        # Using a structured template via langchain_core
        template = "Convert these raw voice notes into clear, bulleted study points:\n\n{text}"
        prompt = PromptTemplate.from_template(template)
        chain = prompt | llm
        response = chain.invoke({"text": chunk})
        summaries.append(response)
    return summaries

def format_notes(summaries):
    """Combines all summaries into a final formatted string."""
    return "\n\n".join(summaries)


In [ ]:
import os

# --- 1. SET YOUR FILE PATH ---
# If you are in Colab, right-click your file in the sidebar and 'Copy path'
FILE_PATH = "/content/voice-sample.txt"

# --- 2. EXECUTION BLOCK ---
if os.path.exists(FILE_PATH):
    # Read the text file directly
    with open(FILE_PATH, 'r', encoding='utf-8') as f:
        content = f.read()

    print(f"✅ Text loaded successfully! ({len(content)} characters)")
    print("✨ Processing with Llama3...")

    # --- 3. RUN YOUR AGENTS ---
    # This part sends the text to your LangChain/Ollama logic
    try:
        split_output = split_content(content)
        summarized_output = summarize_section(split_output)
        final_notes = format_notes(summarized_output)

        print("\n✅ Final Study Notes:\n", final_notes)
    except NameError as e:
        print(f"❌ Error: One of your agent functions is missing: {e}")
else:
    print(f"❌ Error: The file '{FILE_PATH}' was not found. Please check the path in the sidebar.")


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
def generate_flashcards(text: str) -> str:
    print("🗂️ Generating flashcards...")
    messages = [
        SystemMessage(content="""You are a study expert. Create 5-10 high-quality flashcards from the text.
        Format them exactly like this:
        Front: [Question or Term]
        Back: [Answer or Definition]
        ---"""),
        HumanMessage(content=text)
    ]
    return llm.invoke(messages)


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage # <--- FIX IS HERE

# Initialize LLM
llm = OllamaLLM(model="llama3.2")

def split_content(text: str):
    messages = [
        SystemMessage(content="Split the text into 3–5 logical topics."),
        HumanMessage(content=text)
    ]
    return llm.invoke(messages)

def summarize_section(section: str):
    messages = [
        SystemMessage(content="Summarize this section into 3–5 key bullet points."),
        HumanMessage(content=section)
    ]
    return llm.invoke(messages)

def format_notes(summaries: str):
    messages = [
        SystemMessage(content="Convert the summaries into clean, bullet-style study notes."),
        HumanMessage(content=summaries)
    ]
    return llm.invoke(messages)

def generate_flashcards(text: str):
    print("🗂️ Generating flashcards...")
    messages = [
        SystemMessage(content="Create 5-10 flashcards. Format: 'Front: [Q] Back: [A]'."),
        HumanMessage(content=text)
    ]
    return llm.invoke(messages)


In [ ]:
# --- FINAL EXECUTION BLOCK ---

if 'content' in locals() or 'content' in globals():
    try:
        print("\n🚀 STARTING COMPLETE ANALYSIS...")

        # 1. Split & Print Topics
        topics_raw = split_content(content)
        print("\n📌 IDENTIFIED TOPICS:\n" + "-"*30)
        print(topics_raw)
        print("-" * 30)

        # 2. Summarize
        summarized_output = summarize_section(topics_raw)

        # 3. Format Study Notes
        final_notes = format_notes(summarized_output)
        print("\n✅ FINAL STUDY NOTES:\n")
        print(final_notes)

        # 4. GENERATE FLASHCARDS (Check this part!)
        # We pass the summarized text to the flashcard agent
        flashcards_output = generate_flashcards(summarized_output)

        print("\n🗂️ FLASHCARDS:\n")
        print("-" * 30)
        print(flashcards_output) # <--- THIS PRINTS THEM TO YOUR SCREEN
        print("-" * 30)

        # 5. SAVE ALL TO ONE FILE
        full_output = f"STUDY NOTES\n{final_notes}\n\nFLASHCARDS\n{flashcards_output}"
        with open("Complete_Study_Packet.txt", "w", encoding='utf-8') as f:
            f.write(full_output)

        print("\n💾 Success! Saved as 'Complete_Study_Packet.txt' in the sidebar.")

    except Exception as e:
        print(f"❌ Error during processing: {e}")
else:
    print("❌ Error: 'content' variable not found. Run the data loading cell first!")
